# Multi-label классификация Telegram-постов по темам ЦБ

Ноутбук подготовлен для запуска на Kaggle GPU. Он обучает ruBERT определять, к каким из тематических посылов относится Telegram-пост:

- `t1_relevant` - ставка высокая надолго;
- `t2_relevant` - таргет 4% без вариантов;
- `t3_relevant` - сберегать сейчас выгодно;
- `t4_relevant` - инфляция от перегретого спроса;
- `t5_relevant` - конец массовой льготной ипотеки.

Это multi-label задача: один пост может относиться сразу к нескольким темам. Поэтому используется sigmoid по каждой теме, `BCEWithLogitsLoss`, отдельный `pos_weight` на каждую тему и отдельный threshold для каждой темы.

В этом варианте добавлены улучшения для следующего эксперимента: переключатель `base`/`large`, менее агрессивный `sqrt` pos_weight, dynamic padding, выбор checkpoint по `macro_pr_auc`, gradient checkpointing для large и отдельный error analysis для проблемных `t3_relevant`/`t4_relevant`.

Важно: этот ноутбук не решает binary relevance задачу `relevant / not relevant`. Колонка `relevant` используется только для sanity-check, потому что по документации она равна `OR(t1..t5)`.


## 1. Комментарий по документации корпуса

Перед правками учтена документация `data/annotated_data/doc.md` и локально проверены parquet-схемы `bert_relevance_corpus_train.parquet` / `bert_relevance_corpus_test.parquet`.

Ключевые решения:

- текстовая колонка: `text`;
- тематические label columns: `t1_relevant` ... `t5_relevant`;
- `t6_relevant` по умолчанию не используется, потому что в собранном BERT-корпусе темы приведены к фиксированным `t1..t5`, а t6 удалена;
- служебные колонки (`channel_id`, `message_id`, `post_date`, `source_file`, `annotated`, `relevant`, `reasoning` и т.п.) не используются как признаки;
- `annotated=False` означает негатив, отсеченный префильтром; среди таких нулей возможен шум;
- основной режим остается `DATA_MODE = "clean"`: train/validation/test берутся только из `annotated=True`, чтобы метки по темам были чище.

`clean` - основной честный режим. `hybrid` можно использовать как дополнительный эксперимент, если нужны отсеченные префильтром негативы только в train. `full` лучше рассматривать как стресс-тест, а не основной результат.


## 2. Опциональная установка библиотек


In [ ]:
# По умолчанию на Kaggle обычно уже есть нужные библиотеки.
# Если окружение пустое, можно раскомментировать строку ниже.
# !pip install -q transformers datasets accelerate scikit-learn


## 3. Импорты и глобальные параметры


In [ ]:
from pathlib import Path
import importlib
import inspect
import json
import os
import platform
import random
import sys
import urllib.request
import warnings

import numpy as np
import pandas as pd

SEED = 42
RANDOM_STATE = 42
VAL_SIZE = 0.15

MODEL_SIZE = "large"  # "base" или "large"
if MODEL_SIZE == "base":
    MODEL_NAME = "ai-forever/ruBert-base"
    PER_DEVICE_TRAIN_BATCH_SIZE = 4
    PER_DEVICE_EVAL_BATCH_SIZE = 4
    GRADIENT_ACCUMULATION_STEPS = 4
    FP16 = True
    GRADIENT_CHECKPOINTING = False
elif MODEL_SIZE == "large":
    MODEL_NAME = "ai-forever/ruBert-large"
    PER_DEVICE_TRAIN_BATCH_SIZE = 1
    PER_DEVICE_EVAL_BATCH_SIZE = 1
    GRADIENT_ACCUMULATION_STEPS = 16
    FP16 = True
    GRADIENT_CHECKPOINTING = True
else:
    raise ValueError("MODEL_SIZE must be 'base' or 'large'")

MAX_LENGTH = 512
USE_T6 = False

POS_WEIGHT_MODE = "sqrt"  # "raw", "sqrt", "capped", "none"
POS_WEIGHT_CAP = 15.0

# "clean"  - использовать только annotated=True;
# "full"   - использовать все строки, включая annotated=False;
# "hybrid" - validation/test только annotated=True, но annotated=False добавить в train.
DATA_MODE = "clean"

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
WORK_DIR.mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)

    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True
    except Exception:
        pass


seed_everything(SEED)
warnings.filterwarnings("default")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)

print("Seed:", SEED)
print("Work dir:", WORK_DIR)
print("MODEL_SIZE:", MODEL_SIZE)
print("MODEL_NAME:", MODEL_NAME)
print("POS_WEIGHT_MODE:", POS_WEIGHT_MODE)
print("DATA_MODE:", DATA_MODE)


## 4. Пути к данным

На Kaggle загрузите `bert_relevance_corpus_train.parquet`, `bert_relevance_corpus_test.parquet` и, по возможности, `doc.md` как Kaggle Dataset.

Замените `<dataset-name>` на имя вашего Kaggle Dataset:

```python
KAGGLE_DATASET_NAME = "telegram-bert-corpus"
```

Если оставить `<dataset-name>`, код попробует найти файлы автоматически внутри `/kaggle/input`. Локальные пути оставлены ровно под структуру проекта.


In [ ]:
IS_KAGGLE = True
KAGGLE_DATASET_NAME = "<dataset-name>"

if IS_KAGGLE:
    TRAIN_PATH = f"/kaggle/input/{KAGGLE_DATASET_NAME}/bert_relevance_corpus_train.parquet"
    TEST_PATH = f"/kaggle/input/{KAGGLE_DATASET_NAME}/bert_relevance_corpus_test.parquet"
    DOC_PATH = f"/kaggle/input/{KAGGLE_DATASET_NAME}/doc.md"
else:
    TRAIN_PATH = "../data/annotated_data/bert_relevance_corpus_train.parquet"
    TEST_PATH = "../data/annotated_data/bert_relevance_corpus_test.parquet"
    DOC_PATH = "../data/annotated_data/doc.md"


def auto_find_file(filename: str) -> str | None:
    root = Path("/kaggle/input")
    if not root.exists():
        return None
    candidates = sorted(root.rglob(filename))
    return str(candidates[0]) if candidates else None


if IS_KAGGLE and KAGGLE_DATASET_NAME == "<dataset-name>":
    auto_train = auto_find_file("bert_relevance_corpus_train.parquet")
    auto_test = auto_find_file("bert_relevance_corpus_test.parquet")
    auto_doc = auto_find_file("doc.md")
    if auto_train and auto_test:
        TRAIN_PATH = auto_train
        TEST_PATH = auto_test
        if auto_doc:
            DOC_PATH = auto_doc
        print("KAGGLE_DATASET_NAME не задан, файлы найдены автоматически.")
    else:
        print("KAGGLE_DATASET_NAME не задан. Замените <dataset-name> на имя Kaggle Dataset.")

print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH:", TEST_PATH)
print("DOC_PATH:", DOC_PATH)


## 5. Проверка среды Kaggle


In [ ]:
def module_version(name: str) -> str:
    try:
        module = importlib.import_module(name)
        return getattr(module, "__version__", "version unknown")
    except Exception as exc:
        return f"not available: {type(exc).__name__}: {exc}"


def check_huggingface(timeout: int = 5) -> str:
    try:
        with urllib.request.urlopen("https://huggingface.co", timeout=timeout) as response:
            return f"available, status={response.status}"
    except Exception as exc:
        return f"not available: {type(exc).__name__}: {exc}"


print("=" * 80)
print("SYSTEM")
print("=" * 80)
print("Python:", sys.version)
print("Platform:", platform.platform())

print("\n" + "=" * 80)
print("LIBRARIES")
print("=" * 80)
for lib in ["torch", "transformers", "datasets", "sklearn", "pandas", "numpy", "pyarrow"]:
    print(f"{lib}:", module_version(lib))

print("\n" + "=" * 80)
print("CUDA")
print("=" * 80)
try:
    import torch
    print("Torch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA version:", torch.version.cuda)
    print("GPU count:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
    if torch.cuda.is_available():
        x = torch.tensor([1.0, 2.0, 3.0], device="cuda")
        y = x * 2
        print("CUDA smoke test:", y)
except Exception as exc:
    print("Torch/CUDA check failed:", repr(exc))

print("\n" + "=" * 80)
print("HUGGING FACE AND FILES")
print("=" * 80)
print("Hugging Face:", check_huggingface())
print("Train exists:", Path(TRAIN_PATH).exists(), TRAIN_PATH)
print("Test exists :", Path(TEST_PATH).exists(), TEST_PATH)
print("Doc exists  :", Path(DOC_PATH).exists(), DOC_PATH)


## 6. Загрузка документации `doc.md`


In [ ]:
doc_path = Path(DOC_PATH)
if doc_path.exists():
    doc_text = doc_path.read_text(encoding="utf-8")
    print(doc_text[:4000])
else:
    doc_text = ""
    print("doc.md не найден по DOC_PATH. Продолжаем с правилами, зафиксированными в markdown ноутбука.")


## 7. Загрузка train/test parquet


In [ ]:
train_raw = pd.read_parquet(TRAIN_PATH)
test_raw = pd.read_parquet(TEST_PATH)

print("train_raw shape:", train_raw.shape)
print("test_raw shape :", test_raw.shape)

print("\nTrain columns:")
print(train_raw.columns.tolist())

print("\nTest columns:")
print(test_raw.columns.tolist())

display(train_raw.head())
display(test_raw.head())


## 8. Текстовая колонка, label columns и служебные колонки


In [ ]:
TEXT_COL = "text"

topic_label_cols = [
    "t1_relevant",
    "t2_relevant",
    "t3_relevant",
    "t4_relevant",
    "t5_relevant",
]

label_cols = topic_label_cols.copy()
if USE_T6:
    if "t6_relevant" in train_raw.columns and "t6_relevant" in test_raw.columns:
        label_cols.append("t6_relevant")
    else:
        raise ValueError("USE_T6=True, но t6_relevant отсутствует в train или test.")
elif "t6_relevant" in train_raw.columns or "t6_relevant" in test_raw.columns:
    print("t6_relevant найден, но не используется: USE_T6=False.")

possible_id_cols = [
    "post_uid",
    "channel_id",
    "channel_name",
    "channel_username",
    "message_id",
    "post_id",
    "post_date",
    "source_file",
]

service_cols = [
    "relevant",
    "annotated",
    "reasoning",
    "source",
    "text_hash",
]


def add_post_uid_if_possible(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "post_uid" not in out.columns and {"channel_id", "message_id"}.issubset(out.columns):
        out["post_uid"] = out["channel_id"].astype(str) + "_" + out["message_id"].astype(str)
    return out


train_raw = add_post_uid_if_possible(train_raw)
test_raw = add_post_uid_if_possible(test_raw)

id_cols = [col for col in possible_id_cols if col in train_raw.columns and col in test_raw.columns]
analysis_cols = []
for col in id_cols + service_cols:
    if col in train_raw.columns and col in test_raw.columns and col not in analysis_cols:
        analysis_cols.append(col)

print("TEXT_COL:", TEXT_COL)
print("label_cols:", label_cols)
print("id_cols:", id_cols)
print("analysis_cols:", analysis_cols)


## 9. Проверка схемы данных

На этом шаге проверяем, что есть текст, все тематические метки, служебные идентификаторы и что `relevant` согласуется с `OR(t1..t5)`. `relevant` не используется как target.


In [ ]:
def validate_schema(df: pd.DataFrame, split_name: str) -> None:
    required = [TEXT_COL] + label_cols
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"{split_name}: отсутствуют обязательные колонки: {missing}")

    empty_texts = int(df[TEXT_COL].fillna("").astype(str).str.strip().eq("").sum())
    duplicate_count = int(df.duplicated(id_cols).sum()) if id_cols else 0

    print(f"\n{split_name}")
    print("rows:", len(df))
    print("empty texts:", empty_texts)
    print("duplicates by id_cols:", duplicate_count if id_cols else "skipped, id_cols empty")

    if "relevant" in df.columns:
        topic_or = (df[label_cols].fillna(False).astype(int).sum(axis=1) > 0).astype(int)
        relevant_values = df["relevant"].fillna(0).astype(int)
        mismatch = int((topic_or != relevant_values).sum())
        print("relevant vs OR(topic labels) mismatches:", mismatch)

    if "annotated" in df.columns:
        print("annotated distribution:")
        print(df["annotated"].value_counts(dropna=False))


validate_schema(train_raw, "train_raw")
validate_schema(test_raw, "test_raw")


## 10. Preprocessing

Labels приводятся к `float32` до создания Hugging Face Dataset. Это важно для `BCEWithLogitsLoss`: если labels останутся `int`/`Long`, при обучении можно получить ошибку приведения типов.


In [ ]:
def prepare_dataframe(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    before_rows = len(df)
    out = df.copy()

    missing = [col for col in [TEXT_COL] + label_cols if col not in out.columns]
    if missing:
        raise ValueError(f"{split_name}: отсутствуют обязательные колонки: {missing}")

    empty_before = int(out[TEXT_COL].fillna("").astype(str).str.strip().eq("").sum())

    keep_cols = []
    for col in [TEXT_COL] + label_cols + analysis_cols:
        if col in out.columns and col not in keep_cols:
            keep_cols.append(col)
    out = out[keep_cols].copy()

    out[TEXT_COL] = out[TEXT_COL].fillna("").astype(str).str.strip()
    out = out[out[TEXT_COL] != ""].copy()

    out[label_cols] = out[label_cols].fillna(False).astype("float32")
    out["labels"] = out[label_cols].values.tolist()
    out["has_any_topic"] = (out[label_cols].sum(axis=1) > 0).astype(int)
    out["num_topics"] = out[label_cols].sum(axis=1).astype(int)

    print(f"\n{split_name}")
    print("rows before:", before_rows)
    print("empty texts before:", empty_before)
    print("rows after:", len(out))
    print("has_any_topic distribution:")
    print(out["has_any_topic"].value_counts(dropna=False).sort_index())
    print("per-topic positives:")
    print(out[label_cols].sum().astype(int))
    print("multi-label posts:", int((out["num_topics"] > 1).sum()))

    return out.reset_index(drop=True)


train_full_df = prepare_dataframe(train_raw, "train_full_df")
test_full_df = prepare_dataframe(test_raw, "test_full_df")


## 11. Режим clean/full/hybrid с учетом `annotated`


In [ ]:
def apply_data_mode(train_df_all: pd.DataFrame, test_df_all: pd.DataFrame, mode: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    mode = mode.lower().strip()
    if mode not in {"clean", "full", "hybrid"}:
        raise ValueError("DATA_MODE должен быть clean, full или hybrid.")

    has_annotated = "annotated" in train_df_all.columns and "annotated" in test_df_all.columns
    if not has_annotated:
        print("Колонка annotated отсутствует. DATA_MODE фактически работает как full.")
        return train_df_all.copy(), pd.DataFrame(), test_df_all.copy()

    train_annotated = train_df_all[train_df_all["annotated"].astype(bool)].copy()
    train_unannotated = train_df_all[~train_df_all["annotated"].astype(bool)].copy()
    test_annotated = test_df_all[test_df_all["annotated"].astype(bool)].copy()

    if mode == "clean":
        return train_annotated, pd.DataFrame(), test_annotated
    if mode == "hybrid":
        return train_annotated, train_unannotated, test_annotated
    return train_df_all.copy(), pd.DataFrame(), test_df_all.copy()


split_source_df, extra_train_df, test_df = apply_data_mode(train_full_df, test_full_df, DATA_MODE)

print("DATA_MODE:", DATA_MODE)
print("split_source_df:", split_source_df.shape)
print("extra_train_df :", extra_train_df.shape)
print("test_df        :", test_df.shape)


## 12. Train/validation split


In [ ]:
from sklearn.model_selection import train_test_split

stratify_values = split_source_df["has_any_topic"]
if stratify_values.nunique() < 2 or stratify_values.value_counts().min() < 2:
    print("Стратификация по has_any_topic невозможна, используем stratify=None.")
    stratify_values = None

train_df, val_df = train_test_split(
    split_source_df,
    test_size=VAL_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=stratify_values,
)

if DATA_MODE == "hybrid" and len(extra_train_df):
    train_df = pd.concat([train_df, extra_train_df], ignore_index=True)
    train_df = train_df.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
else:
    train_df = train_df.reset_index(drop=True)

val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("train_df:", train_df.shape)
print("val_df  :", val_df.shape)
print("test_df :", test_df.shape)


## 13. Распределение тем по split


In [ ]:
def show_label_distribution(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    rows = []
    total = len(df)
    any_count = int(df["has_any_topic"].sum())
    multi_count = int((df["num_topics"] > 1).sum())

    print(f"\n{split_name}")
    print("rows:", total)
    print(f"has_any_topic: {any_count} ({any_count / total:.2%})")
    print(f"multi-label posts: {multi_count} ({multi_count / total:.2%})")

    for label in label_cols:
        positive = int(df[label].sum())
        rows.append({
            "split": split_name,
            "label": label,
            "positive_count": positive,
            "positive_share": positive / total,
        })

    result = pd.DataFrame(rows)
    display(result)
    return result


dist_train = show_label_distribution(train_df, "train")
dist_val = show_label_distribution(val_df, "validation")
dist_test = show_label_distribution(test_df, "test")

distribution_all = pd.concat([dist_train, dist_val, dist_test], ignore_index=True)
distribution_all.to_csv(WORK_DIR / "label_distribution_splits.csv", index=False, encoding="utf-8-sig")


## 14. Проверка пересечений train/validation/test


In [ ]:
def overlap_count(left: pd.DataFrame, right: pd.DataFrame, name_left: str, name_right: str) -> int | None:
    if "post_uid" not in left.columns or "post_uid" not in right.columns:
        print(f"{name_left} vs {name_right}: overlap check skipped, post_uid отсутствует.")
        return None
    overlap = set(left["post_uid"].astype(str)) & set(right["post_uid"].astype(str))
    print(f"{name_left} vs {name_right}: overlap={len(overlap)}")
    return len(overlap)


overlap_count(train_df, val_df, "train", "validation")
overlap_count(train_df, test_df, "train", "test")
overlap_count(val_df, test_df, "validation", "test")


## 15. Загрузка tokenizer ruBERT (`base` / `large`)


In [ ]:
import torch
from datasets import Dataset, Features, Sequence, Value
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer loaded:", MODEL_NAME)


## 16. Создание Hugging Face Dataset


In [ ]:
def make_hf_dataset(df: pd.DataFrame) -> Dataset:
    data = df[[TEXT_COL, "labels"]].copy()
    features = Features({
        TEXT_COL: Value("string"),
        "labels": Sequence(Value("float32")),
    })
    return Dataset.from_pandas(data, preserve_index=False, features=features)


train_dataset = make_hf_dataset(train_df)
val_dataset = make_hf_dataset(val_df)
test_dataset = make_hf_dataset(test_df)

print(train_dataset)
print(val_dataset)
print(test_dataset)


## 17. Токенизация с dynamic padding

Fixed padding до 512 токенов заранее не используется. Вместо этого тексты только обрезаются до `MAX_LENGTH`, а padding выполняет `DataCollatorWithPadding` внутри каждого batch.

Dynamic padding уменьшает расход памяти и ускоряет обучение, потому что каждый batch дополняется только до максимальной длины внутри batch, а не все тексты заранее до 512 токенов. Для `ruBERT-large` это особенно важно.


In [ ]:
def tokenize_batch(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )


train_dataset = train_dataset.map(tokenize_batch, batched=True)
val_dataset = val_dataset.map(tokenize_batch, batched=True)
test_dataset = test_dataset.map(tokenize_batch, batched=True)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
)

print(train_dataset.column_names)


## 18. Torch format и проверка dtype labels


In [ ]:
train_dataset = train_dataset.remove_columns([TEXT_COL])
val_dataset = val_dataset.remove_columns([TEXT_COL])
test_dataset = test_dataset.remove_columns([TEXT_COL])

torch_columns = ["input_ids", "attention_mask", "labels"]
if "token_type_ids" in train_dataset.column_names:
    torch_columns.insert(1, "token_type_ids")

train_dataset.set_format(type="torch", columns=torch_columns)
val_dataset.set_format(type="torch", columns=torch_columns)
test_dataset.set_format(type="torch", columns=torch_columns)

sample = train_dataset[0]
print(sample.keys())
print("input_ids shape:", sample["input_ids"].shape)
print("attention_mask shape:", sample["attention_mask"].shape)
print("labels:", sample["labels"])
print("labels dtype:", sample["labels"].dtype)
print("labels shape:", sample["labels"].shape)

assert sample["labels"].dtype == torch.float32
assert sample["labels"].shape == torch.Size([len(label_cols)])


## 19. Загрузка модели и gradient checkpointing

Для `MODEL_SIZE="large"` включается gradient checkpointing: это снижает расход GPU memory, но делает обучение немного медленнее. Для Kaggle T4 это разумный обмен.


In [ ]:
id2label = {i: label for i, label in enumerate(label_cols)}
label2id = {label: i for i, label in enumerate(label_cols)}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_cols),
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id,
)

if GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable()
    model.config.use_cache = False

print("Model loaded:", MODEL_NAME)
print("MODEL_SIZE:", MODEL_SIZE)
print("num_labels:", model.config.num_labels)
print("gradient_checkpointing:", GRADIENT_CHECKPOINTING)
print("labels:", label_cols)


## 20. Pos weight для дисбаланса

В baseline raw `pos_weight = negative / positive` оказался слишком агрессивным: модель завышала вероятности позитивных классов, а оптимальные thresholds уходили к 0.85-0.95.

По умолчанию используется `POS_WEIGHT_MODE = "sqrt"`: `sqrt(raw_pos_weight)` сохраняет компенсацию дисбаланса, но делает ее мягче. Для сравнения доступны режимы `"raw"`, `"capped"` и `"none"`.


In [ ]:
positive_counts = train_df[label_cols].sum(axis=0).astype(float)
negative_counts = len(train_df) - positive_counts

raw_pos_weight = negative_counts / positive_counts.replace(0, np.nan)
raw_pos_weight = raw_pos_weight.replace([np.inf, -np.inf], np.nan).fillna(1.0)

if POS_WEIGHT_MODE == "raw":
    pos_weight_values = raw_pos_weight
elif POS_WEIGHT_MODE == "sqrt":
    pos_weight_values = np.sqrt(raw_pos_weight)
elif POS_WEIGHT_MODE == "capped":
    pos_weight_values = raw_pos_weight.clip(upper=POS_WEIGHT_CAP)
elif POS_WEIGHT_MODE == "none":
    pos_weight_values = pd.Series(1.0, index=label_cols)
else:
    raise ValueError("Unknown POS_WEIGHT_MODE")

pos_weight = torch.tensor(pos_weight_values.values, dtype=torch.float32)

label_distribution_train = pd.DataFrame({
    "label": label_cols,
    "positive_count": positive_counts.values.astype(int),
    "negative_count": negative_counts.values.astype(int),
    "raw_pos_weight": raw_pos_weight.values,
    "used_pos_weight": pos_weight_values.values,
})
display(label_distribution_train)
label_distribution_train.to_csv(WORK_DIR / "label_distribution_train.csv", index=False, encoding="utf-8-sig")


## 21. Метрики для multi-label задачи

F1/precision/recall при threshold 0.5 остаются для мониторинга. Для выбора лучшего checkpoint используется `macro_pr_auc`: эта метрика не зависит от фиксированного threshold 0.5 и лучше согласуется с последующим threshold tuning по validation.


In [ ]:
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)


def sigmoid_np(x):
    x = np.asarray(x)
    return 1 / (1 + np.exp(-x))


def safe_average_precision(y_true, y_score):
    y_true = np.asarray(y_true)
    if np.sum(y_true) == 0:
        return np.nan
    return average_precision_score(y_true, y_score)


def multilabel_metrics(labels: np.ndarray, preds: np.ndarray) -> dict:
    return {
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
        "micro_precision": precision_score(labels, preds, average="micro", zero_division=0),
        "macro_precision": precision_score(labels, preds, average="macro", zero_division=0),
        "micro_recall": recall_score(labels, preds, average="micro", zero_division=0),
        "macro_recall": recall_score(labels, preds, average="macro", zero_division=0),
    }


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]

    labels = np.asarray(labels).astype(int)
    probs = sigmoid_np(np.asarray(logits))
    preds = (probs >= 0.5).astype(int)

    metrics = multilabel_metrics(labels, preds)

    per_label_ap = []
    for i, label in enumerate(label_cols):
        ap = safe_average_precision(labels[:, i], probs[:, i])
        if not np.isnan(ap):
            ap = float(ap)
            per_label_ap.append(ap)
            metrics[f"pr_auc_{label}"] = ap

    metrics["macro_pr_auc"] = float(np.mean(per_label_ap)) if per_label_ap else 0.0

    try:
        metrics["micro_pr_auc"] = float(average_precision_score(labels.ravel(), probs.ravel()))
    except Exception:
        metrics["micro_pr_auc"] = 0.0

    return metrics


## 22. WeightedMultilabelTrainer


In [ ]:
from transformers import Trainer


class WeightedMultilabelTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").float()
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = torch.nn.BCEWithLogitsLoss(
            pos_weight=self.pos_weight.to(logits.device) if self.pos_weight is not None else None
        )

        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


## 23. TrainingArguments для `base` / `large`

Для `large` используются batch size 1, gradient accumulation 16, fp16, gradient checkpointing и `adamw_torch`. Для `base` настройки мягче и быстрее. Лучший checkpoint выбирается по `macro_pr_auc`, а не по `macro_f1` при threshold 0.5.


In [ ]:
from transformers import TrainingArguments

# If GPU memory is enough for large, try:
# PER_DEVICE_TRAIN_BATCH_SIZE = 2
# PER_DEVICE_EVAL_BATCH_SIZE = 2
# GRADIENT_ACCUMULATION_STEPS = 8

training_kwargs = {
    "output_dir": str(WORK_DIR / f"{MODEL_SIZE}_topic_multilabel"),
    "save_strategy": "epoch",
    "learning_rate": 2e-5,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "num_train_epochs": 4,
    "weight_decay": 0.01,
    "load_best_model_at_end": True,
    "metric_for_best_model": "macro_pr_auc",
    "greater_is_better": True,
    "logging_steps": 50,
    "save_total_limit": 2,
    "report_to": "none",
    "fp16": FP16,
    "gradient_checkpointing": GRADIENT_CHECKPOINTING,
    "optim": "adamw_torch",
    "seed": SEED,
    "data_seed": SEED,
}

# Если fp16=True вызывает проблемы в конкретном окружении, поставьте:
# training_kwargs["fp16"] = False

training_signature = inspect.signature(TrainingArguments.__init__).parameters
if "eval_strategy" in training_signature:
    training_kwargs["eval_strategy"] = "epoch"
else:
    training_kwargs["evaluation_strategy"] = "epoch"

unsupported_args = sorted(key for key in training_kwargs if key not in training_signature)
if unsupported_args:
    print("TrainingArguments: unsupported args skipped:", unsupported_args)
    training_kwargs = {key: value for key, value in training_kwargs.items() if key in training_signature}

training_args = TrainingArguments(**training_kwargs)
print(training_args)


## 24. Создание Trainer


In [ ]:
trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_dataset,
    "eval_dataset": val_dataset,
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
    "pos_weight": pos_weight,
}

trainer_signature = inspect.signature(Trainer.__init__).parameters
if "processing_class" in trainer_signature:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_signature:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = WeightedMultilabelTrainer(**trainer_kwargs)
print("Trainer готов")


## 25. Обучение

Эту ячейку запускайте на Kaggle GPU. Локально ноутбук не обучался.


In [ ]:
train_result = trainer.train()

trainer.save_metrics("train", train_result.metrics)
trainer.save_state()

with open(WORK_DIR / "train_metrics.json", "w", encoding="utf-8") as f:
    json.dump(train_result.metrics, f, ensure_ascii=False, indent=2)

print(train_result.metrics)


## 26. Validation evaluation при threshold 0.5


In [ ]:
val_metrics = trainer.evaluate(eval_dataset=val_dataset)
print(val_metrics)

with open(WORK_DIR / "validation_metrics_raw_threshold_05.json", "w", encoding="utf-8") as f:
    json.dump(val_metrics, f, ensure_ascii=False, indent=2)


## 27. Threshold tuning по validation

Threshold подбирается отдельно для каждой темы только на validation. Test не используется для выбора порогов.


In [ ]:
val_pred = trainer.predict(val_dataset)
val_logits = val_pred.predictions
val_labels = val_pred.label_ids.astype(int)
val_probs = sigmoid_np(val_logits)

threshold_grid = np.arange(0.05, 0.96, 0.05)

threshold_rows = []
best_thresholds = {}

for i, label in enumerate(label_cols):
    y_true = val_labels[:, i]
    best_row = None

    for threshold in threshold_grid:
        y_pred = (val_probs[:, i] >= threshold).astype(int)
        precision, recall, f1, support = precision_recall_fscore_support(
            y_true,
            y_pred,
            average="binary",
            zero_division=0,
        )
        row = {
            "label": label,
            "threshold": round(float(threshold), 4),
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "support": int(y_true.sum()),
        }
        threshold_rows.append(row)

        if best_row is None:
            best_row = row
        else:
            current_key = (row["f1"], row["recall"], -row["threshold"])
            best_key = (best_row["f1"], best_row["recall"], -best_row["threshold"])
            if current_key > best_key:
                best_row = row

    best_thresholds[label] = float(best_row["threshold"])

threshold_search_df = pd.DataFrame(threshold_rows)
display(threshold_search_df.sort_values(["label", "f1"], ascending=[True, False]).groupby("label").head(5))

with open(WORK_DIR / "best_thresholds.json", "w", encoding="utf-8") as f:
    json.dump(best_thresholds, f, ensure_ascii=False, indent=2)

threshold_search_df.to_csv(WORK_DIR / "validation_threshold_search.csv", index=False, encoding="utf-8-sig")

print("best_thresholds:")
print(json.dumps(best_thresholds, ensure_ascii=False, indent=2))


## 28. Test evaluation с tuned thresholds

Финальная оценка считается на test с thresholds, подобранными только на validation. В `topic_metrics.json` сохраняются параметры эксперимента, включая `MODEL_SIZE`, `DATA_MODE`, `POS_WEIGHT_MODE` и метрику выбора checkpoint.


In [ ]:
test_pred = trainer.predict(test_dataset)
test_logits = test_pred.predictions
test_labels = test_pred.label_ids.astype(int)
test_probs = sigmoid_np(test_logits)

thresholds_array = np.array([best_thresholds[label] for label in label_cols])
test_preds = (test_probs >= thresholds_array).astype(int)

test_metrics = multilabel_metrics(test_labels, test_preds)

topic_metrics = {
    "model_name": MODEL_NAME,
    "model_size": MODEL_SIZE,
    "data_mode": DATA_MODE,
    "pos_weight_mode": POS_WEIGHT_MODE,
    "pos_weight_cap": POS_WEIGHT_CAP,
    "max_length": MAX_LENGTH,
    "metric_for_best_model": "macro_pr_auc",
    "thresholds": best_thresholds,
    "labels": label_cols,
    "test_rows": int(len(test_df)),
    "test_metrics": test_metrics,
}

print(json.dumps(topic_metrics, ensure_ascii=False, indent=2))

with open(WORK_DIR / "topic_metrics.json", "w", encoding="utf-8") as f:
    json.dump(topic_metrics, f, ensure_ascii=False, indent=2)

report = classification_report(
    test_labels,
    test_preds,
    target_names=label_cols,
    zero_division=0,
    output_dict=True,
)
report_df = pd.DataFrame(report).transpose()
display(report_df)
report_df.to_csv(WORK_DIR / "topic_test_classification_report.csv", encoding="utf-8-sig")


## 29. Таблица качества по каждой теме


In [ ]:
per_topic_rows = []
for i, label in enumerate(label_cols):
    precision, recall, f1, support = precision_recall_fscore_support(
        test_labels[:, i],
        test_preds[:, i],
        average="binary",
        zero_division=0,
    )
    per_topic_rows.append({
        "label": label,
        "support": int(test_labels[:, i].sum()),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "threshold": float(best_thresholds[label]),
        "train_positives": int(train_df[label].sum()),
        "val_positives": int(val_df[label].sum()),
        "test_positives": int(test_df[label].sum()),
        "raw_pos_weight": float(raw_pos_weight[label]),
        "used_pos_weight": float(pos_weight_values[label]),
    })

per_topic_metrics = pd.DataFrame(per_topic_rows)
display(per_topic_metrics)
per_topic_metrics.to_csv(WORK_DIR / "per_topic_metrics.csv", index=False, encoding="utf-8-sig")


## 30. Export predictions


In [ ]:
prediction_cols = [col for col in id_cols + [TEXT_COL] + label_cols if col in test_df.columns]
test_predictions_df = test_df[prediction_cols].copy()

for i, label in enumerate(label_cols):
    test_predictions_df[f"{label}_prob"] = test_probs[:, i]
    test_predictions_df[f"{label}_pred"] = test_preds[:, i]

test_predictions_df["num_true_topics"] = test_labels.sum(axis=1).astype(int)
test_predictions_df["num_pred_topics"] = test_preds.sum(axis=1).astype(int)
test_predictions_df["any_true_topic"] = (test_predictions_df["num_true_topics"] > 0).astype(int)
test_predictions_df["any_pred_topic"] = (test_predictions_df["num_pred_topics"] > 0).astype(int)

display(test_predictions_df.head())
test_predictions_df.to_csv(WORK_DIR / "topic_test_predictions.csv", index=False, encoding="utf-8-sig")


## 31. Error analysis

В baseline хуже всего работали `t3_relevant` и `t4_relevant`, поэтому для них дополнительно сохраняется отдельный error analysis. В общих CSV остаются ошибки по всем темам.


In [ ]:
error_columns = [
    "label",
    "error_type",
    "text",
    "true_label",
    "predicted_label",
    "probability",
    "threshold",
] + id_cols

false_positive_rows = []
false_negative_rows = []

for i, label in enumerate(label_cols):
    threshold = float(best_thresholds[label])
    for row_idx in range(len(test_df)):
        true_label = int(test_labels[row_idx, i])
        predicted_label = int(test_preds[row_idx, i])
        probability = float(test_probs[row_idx, i])

        if true_label == predicted_label:
            continue

        row = {
            "label": label,
            "error_type": "FP" if true_label == 0 else "FN",
            "text": test_df.iloc[row_idx][TEXT_COL],
            "true_label": true_label,
            "predicted_label": predicted_label,
            "probability": probability,
            "threshold": threshold,
        }
        for col in id_cols:
            if col in test_df.columns:
                row[col] = test_df.iloc[row_idx][col]

        if true_label == 0 and predicted_label == 1:
            false_positive_rows.append(row)
        elif true_label == 1 and predicted_label == 0:
            false_negative_rows.append(row)

false_positives_df = pd.DataFrame(false_positive_rows, columns=error_columns)
false_negatives_df = pd.DataFrame(false_negative_rows, columns=error_columns)

false_positives_df.to_csv(WORK_DIR / "error_analysis_false_positives.csv", index=False, encoding="utf-8-sig")
false_negatives_df.to_csv(WORK_DIR / "error_analysis_false_negatives.csv", index=False, encoding="utf-8-sig")

problem_labels = ["t3_relevant", "t4_relevant"]
t3_t4_false_positives_df = false_positives_df[false_positives_df["label"].isin(problem_labels)].copy()
t3_t4_false_negatives_df = false_negatives_df[false_negatives_df["label"].isin(problem_labels)].copy()

t3_t4_false_positives_df.to_csv(
    WORK_DIR / "error_analysis_t3_t4_false_positives.csv",
    index=False,
    encoding="utf-8-sig",
)
t3_t4_false_negatives_df.to_csv(
    WORK_DIR / "error_analysis_t3_t4_false_negatives.csv",
    index=False,
    encoding="utf-8-sig",
)

print("False positives:", false_positives_df.shape)
print("False negatives:", false_negatives_df.shape)
print("t3/t4 false positives:", t3_t4_false_positives_df.shape)
print("t3/t4 false negatives:", t3_t4_false_negatives_df.shape)

for label in problem_labels:
    print(f"\n===== {label}: false positives =====")
    if len(false_positives_df):
        display(false_positives_df[false_positives_df["label"] == label].sort_values("probability", ascending=False).head(10))
    else:
        print("нет")

    print(f"===== {label}: false negatives =====")
    if len(false_negatives_df):
        display(false_negatives_df[false_negatives_df["label"] == label].sort_values("probability", ascending=True).head(10))
    else:
        print("нет")


## 32. Сохранение финальной модели


In [ ]:
FINAL_MODEL_DIR = WORK_DIR / f"rubert_{MODEL_SIZE}_topic_multilabel_{POS_WEIGHT_MODE}_final"

trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

topic_model_config = {
    "model_name": MODEL_NAME,
    "model_size": MODEL_SIZE,
    "task": "multi_label_topic_classification",
    "labels": label_cols,
    "text_col": TEXT_COL,
    "max_length": MAX_LENGTH,
    "thresholds": best_thresholds,
    "use_t6": USE_T6,
    "data_mode": DATA_MODE,
    "pos_weight_mode": POS_WEIGHT_MODE,
    "pos_weight_cap": POS_WEIGHT_CAP,
    "metric_for_best_model": "macro_pr_auc",
    "train_path": TRAIN_PATH,
    "test_path": TEST_PATH,
    "val_size": VAL_SIZE,
    "random_state": SEED,
}

with open(FINAL_MODEL_DIR / "topic_model_config.json", "w", encoding="utf-8") as f:
    json.dump(topic_model_config, f, ensure_ascii=False, indent=2)

print("Final model saved to:", FINAL_MODEL_DIR)


## 33. Краткий итог эксперимента

Рекомендуемый основной запуск:

```python
MODEL_SIZE = "large"
DATA_MODE = "clean"
POS_WEIGHT_MODE = "sqrt"
MAX_LENGTH = 512
```

После полного запуска на Kaggle в `/kaggle/working` должны появиться:

- `rubert_large_topic_multilabel_sqrt_final/`
- `topic_metrics.json`
- `topic_test_classification_report.csv`
- `per_topic_metrics.csv`
- `topic_test_predictions.csv`
- `validation_threshold_search.csv`
- `best_thresholds.json`
- `label_distribution_train.csv`
- `error_analysis_false_positives.csv`
- `error_analysis_false_negatives.csv`
- `error_analysis_t3_t4_false_positives.csv`
- `error_analysis_t3_t4_false_negatives.csv`

Методологически важное:

- это multi-label классификация, а не binary relevance;
- sigmoid используется вместо softmax, потому что темы независимы и могут встречаться вместе;
- `BCEWithLogitsLoss` подходит для multi-label классификации и стабильнее, чем ручной sigmoid + BCE;
- labels должны быть `float32`, потому что BCE считает loss по float-таргетам;
- `sqrt` pos_weight выбран вместо raw, чтобы компенсировать дисбаланс менее агрессивно;
- `macro_pr_auc` лучше для выбора checkpoint, чем `macro_f1` при фиксированном threshold 0.5;
- thresholds подбираются отдельно по темам на validation, а test остается только для финальной оценки;
- dynamic padding снижает расход памяти для `ruBERT-large`;
- gradient checkpointing для `large` снижает расход GPU memory ценой скорости;
- основной режим данных - `clean`, потому что `annotated=False` содержит шумные негативы.
